# Spherical Shallow-Water Dynamics (Dedalus)

`ShallowWaterSphereDedalus` integrates shallow-water dynamics on a rotating sphere with [Dedalus](https://dedalus-project.org/). This notebook keeps three cases separate so the caveats stay visible:

1. **Dedalus Galewsky example**: the barotropically unstable mid-latitude jet from the official Dedalus sphere IVP example. This is the baseline we should be able to recognize visually.
2. **PDEArena-inspired random jet**: a Dedalus-based random-jet ensemble inspired by PDEArena's shallow-water data generation, not the actual PDEArena generator.
3. **The Well / PlanetSWE-inspired forcing**: the same Dedalus backend with a simple diurnal/seasonal height-tendency forcing shaped like PlanetSWE's documented forcing, not the released dataset pipeline.

Outputs are `[h, u_phi, u_theta]` on a `[longitude, colatitude]` grid. `h` is the height perturbation in metres; velocities are in m/s. `u_theta` is the colatitude component and is positive southward, so northward meridional velocity is `-u_theta`.

For the Dedalus comparison we also compute an approximate relative-vorticity diagnostic from the returned velocities. The official Dedalus script saves vorticity directly with `-div(skew(u))`; computing it here is only for visualization.

In [ ]:
import importlib.util
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML

from autosim.experimental.simulations import ShallowWaterSphereDedalus
from autosim.experimental.simulations.shallow_water_sphere_dedalus import EARTH_RADIUS_M
from autosim.utils import plot_spatiotemporal_video

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_MAX_THREADS", "1")

dedalus_available = importlib.util.find_spec("dedalus") is not None
if not dedalus_available:
    print(
        "Dedalus is not installed. See the README optional spherical "
        "backend notes before running `uv sync --extra spherical`."
    )

## 1. Dedalus Galewsky Example

This section tracks the official [Dedalus spherical shallow-water example](https://dedalus-project.readthedocs.io/en/latest/pages/examples/ivp_sphere_shallow_water.html): `Nphi=256`, `Ntheta=128`, `umax=80 m/s`, `H=1e4 m`, `hpert=120 m`, `timestep=600 s`, and `stop_sim_time=360 h`. The only notebook simplification below is `dt_save=6 h` to keep the in-memory trajectory and animation small; use `dt_save=1.0` to match the example's hourly analysis output.

Expected differences from the official plot:

- The official script writes Dedalus analysis tasks (`height` and vorticity) to HDF5 and plots from those files; here we return `h, u_phi, u_theta` tensors and compute vorticity approximately for display.
- The notebook renders with Matplotlib directly rather than Dedalus' `plot_sphere.py` helper.
- If you reduce `nlon/nlat` for speed, the vortex roll-up will look smoother and less like the reference.

In [ ]:
if dedalus_available:
    sim_dedalus = ShallowWaterSphereDedalus(
        return_timeseries=True,
        log_level="warning",
        nlon=256,
        nlat=128,
        T=360.0,
        dt_save=6.0,  # use 1.0 for hourly output matching the Dedalus example
        timestep_seconds=600.0,
        hyperviscosity_match_mode=32,
        parameters_range={"height_perturbation_m": (120.0, 120.0)},
    )
    batch_dedalus = sim_dedalus.forward_samples_spatiotemporal(n=1, random_seed=7)
    data_dedalus = batch_dedalus["data"][0]
    data_dedalus_np = data_dedalus.detach().cpu().numpy()
    print(data_dedalus.shape)  # [time, longitude, colatitude, channels]

In [ ]:
def relative_vorticity(field: np.ndarray, radius_m: float = EARTH_RADIUS_M) -> np.ndarray:
    """Approximate vertical relative vorticity from [*, lon, colat, channels]."""
    u_phi = np.asarray(field[..., 1])
    u_theta = np.asarray(field[..., 2])
    nlon, ncolat = u_phi.shape[-2:]
    theta = ((np.arange(ncolat) + 0.5) * np.pi / ncolat)[::-1]  # south -> north
    sin_theta = np.sin(theta).reshape((1,) * (u_phi.ndim - 1) + (ncolat,))
    edge_order = 2 if ncolat > 2 else 1
    d_sin_u_dtheta = np.gradient(sin_theta * u_phi, theta, axis=-1, edge_order=edge_order)
    dphi = 2.0 * np.pi / nlon
    d_utheta_dphi = (np.roll(u_theta, -1, axis=-2) - np.roll(u_theta, 1, axis=-2)) / (2.0 * dphi)
    return (d_sin_u_dtheta - d_utheta_dphi) / (radius_m * sin_theta)


def plot_equirectangular(field2d, ax, *, title, cmap="RdBu_r", symmetric=True):
    field2d = np.asarray(field2d)
    if symmetric:
        limit = float(np.nanmax(np.abs(field2d))) or 1.0
        vmin, vmax = -limit, limit
    else:
        vmin, vmax = float(np.nanmin(field2d)), float(np.nanmax(field2d))
    im = ax.imshow(
        field2d.T,
        origin="lower",  # colatitude axis is stored south -> north
        aspect="auto",
        extent=(0.0, 360.0, -90.0, 90.0),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(title)
    ax.set_xlabel("longitude [deg]")
    ax.set_ylabel("latitude [deg]")
    return im

In [ ]:
if dedalus_available:
    vort_dedalus = relative_vorticity(data_dedalus_np)
    h_final = data_dedalus_np[-1, :, :, 0]
    vort_final = vort_dedalus[-1]

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), constrained_layout=True)
    im0 = plot_equirectangular(h_final, axes[0], title="h at 360 h [m]", cmap="coolwarm", symmetric=False)
    im1 = plot_equirectangular(vort_final, axes[1], title="relative vorticity at 360 h [1/s]")
    fig.colorbar(im0, ax=axes[0], shrink=0.85)
    fig.colorbar(im1, ax=axes[1], shrink=0.85)
    plt.show()

In [ ]:
if dedalus_available:
    vort_dedalus_batch = torch.from_numpy(vort_dedalus[None, ..., None]).float()
    anim_dedalus = plot_spatiotemporal_video(
        true=vort_dedalus_batch,
        pred=None,
        batch_idx=0,
        channel_names=["relative vorticity"],
        projection="sphere",
        cmap="RdBu_r",
        title="Dedalus Galewsky vorticity",
    )
    HTML(anim_dedalus.to_jshtml())

## 2. PDEArena-Inspired Random Jet

PDEArena's shallow-water data are generated with the PDEArena data pipeline and SpeedyWeather.jl. The run below is only a Python/Dedalus stand-in: it uses the same spherical Dedalus equations as the first section, but swaps the single Galewsky jet for a seeded low-order random zonal jet and a wave perturbation.

What would be needed to align with PDEArena:

- Use PDEArena's actual generation code/configs and SpeedyWeather.jl rather than Dedalus.
- Match its grid, random initial-condition distribution, variable conventions, time sampling, train/validation/test split, and normalization pipeline.
- Validate against downloaded PDEArena trajectories rather than visual similarity alone.

In [ ]:
if dedalus_available:
    sim_pdearena = ShallowWaterSphereDedalus(
        return_timeseries=True,
        log_level="warning",
        nlon=128,
        nlat=64,
        T=144.0,
        dt_save=3.0,
        timestep_seconds=600.0,
        ic_mode="random_jet",
        n_jet_modes=4,
        perturbation_zonal_mode=6,
        umax_mps=60.0,
        hyperviscosity_match_mode=32,
        parameters_range={"height_perturbation_m": (80.0, 160.0)},
    )
    batch_pdearena = sim_pdearena.forward_samples_spatiotemporal(n=2, random_seed=0)
    print(batch_pdearena["data"].shape)

In [ ]:
if dedalus_available:
    anim_pdearena = plot_spatiotemporal_video(
        true=batch_pdearena["data"],
        pred=None,
        batch_idx=0,  # try batch_idx=1 for the second independently drawn jet
        channel_names=sim_pdearena.output_names,
        projection="sphere",
        title="PDEArena-inspired random jet",
    )
    HTML(anim_pdearena.to_jshtml())

## 3. The Well / PlanetSWE-Inspired Forcing

The Well's [PlanetSWE](https://polymathic-ai.org/the_well/datasets/planetswe/) is forced hyperviscous rotating shallow water on a sphere with Earth-like topography, daily/annual periodic forcing, ERA5-derived 500 hPa initial conditions, and a CFL-based step size. It stores 40 trajectories of 3 model years, with 3024 hourly steps on 256×512 images.

The run below keeps only the simplest shared piece: a Dedalus shallow-water model with a diurnal/seasonal height-tendency forcing shaped like the documented PlanetSWE forcing. It is useful for exercising the same broad mechanism, but it is not a dataset reproduction.

What would be needed to align with PlanetSWE:

- Add ERA5 500 hPa initial states and the burn-in/filtering procedure.
- Add realistic topography and match the PlanetSWE mean depth/hyperviscosity choices.
- Run at the dataset grid and duration (`512×256`, 3024 hourly outputs, 40 trajectories).
- Reintroduce a carefully validated CFL-based timestepper if exact generation fidelity is required.

In [ ]:
if dedalus_available:
    sim_planetswe = ShallowWaterSphereDedalus(
        return_timeseries=True,
        log_level="warning",
        nlon=128,
        nlat=64,
        T=360.0,
        dt_save=6.0,
        skip_nt=8,
        timestep_seconds=600.0,
        ic_mode="random_jet",
        n_jet_modes=4,
        perturbation_zonal_mode=6,
        umax_mps=70.0,
        mean_height_m=5500.0,
        hyperviscosity_match_mode=40,
        forcing=True,
        forcing_rate_m_per_hour=4.0,
        forcing_day_hours=24.0,
        forcing_year_hours=1008.0,
        forcing_declination_rad=0.4,
        parameters_range={"height_perturbation_m": (80.0, 160.0)},
    )
    batch_planetswe = sim_planetswe.forward_samples_spatiotemporal(n=1, random_seed=2)
    print(batch_planetswe["data"].shape)

In [ ]:
if dedalus_available:
    anim_planetswe = plot_spatiotemporal_video(
        true=batch_planetswe["data"],
        pred=None,
        batch_idx=0,
        channel_names=sim_planetswe.output_names,
        projection="sphere",
        title="PlanetSWE-inspired forced run",
    )
    HTML(anim_planetswe.to_jshtml())